# T665118 【MX-J25-T1】「Cfz Round 8」Sqrt Problem

## 题目描述

给定一个整数 $n$，你可以对其进行下面的两种操作：

- $n \leftarrow n+2$，即将 $n$ 增加 $2$。
- 若 $\sqrt n$ 为整数，则 $n \leftarrow \sqrt n$，即将 $n$ 开方。进行此操作后你获得 $1$ 分。

你需要求出获得 $k$ 分所至少需要进行的操作数量。

## 输入格式

**本题包含多组测试数据。**

输入的第一行包含两个非负整数 $c,t$，分别表示测试点编号与测试数据组数。$c=0$ 表示该测试点为样例。

接下来依次输入每组测试数据，对于每组测试数据：

- 共一行，包含两个正整数 $n,k$。

## 输出格式

对于每组测试数据：

- 输出一行，包含一个整数，表示获得 $k$ 分所至少需要进行的操作数量。

## 输入输出样例 #1

### 输入 #1

```
0 5
6 1
1 3
14514 23333
2011112920110906 1
3 1919810233114514
```

### 输出 #1

```
6
3
46860
15268726
7679240932458056
```

## 说明/提示

### 样例 1 解释

本组样例包含 $5$ 组测试数据。

- 对于第 $1$ 组测试数据，依次进行 $5$ 次第 $1$ 种操作和 $1$ 次第 $2$ 种操作即可。可以证明至少需要进行 $6$ 次操作。
- 对于第 $2$ 组测试数据，进行 $3$ 次第 $2$ 种操作即可。可以证明至少需要进行 $3$ 次操作。

### 数据范围

对于所有测试数据，均有：

- $1 \le t \le 10^5$；
- $1 \le n,k \le 10^{18}$。

::cute-table{tuack}

| 测试点编号 | $n \le$ | $k\le$ | 特殊性质 |
| :----------: | :----------: | :----------: | :----------: |
| $1$ | $1$ | $10^5$ | 是 |
| $2$ | ^ | $10^{18}$ | ^ |
| $3$ | $2$ | $10^{5}$ | ^ |
| $4$ | ^ | $10^{18}$ | ^ |
| $5$ | $10^5$ | $1$ | ^ |
| $6$ | $10^{18}$ | ^ | ^ |
| $7$ | $10^5$ | $10^5$ | ^ |
| $8$ | $10^9$ | $10^9$ | ^ |
| $9$ | ^ | ^ | 否 |
| $10$ | $10^{18}$ | $10^{18}$ | ^ |

- 特殊性质：保证 $t=3$。

In [ ]:
import math
def min_operation(n,k):
  ops=0 
  while k>0 and n>1:
    m = math.isqrt(n)
    if m*m ==n:
      n=m
      ops +=1
      k-=1
  
    sq = m+1
    diff=sq*sq-n;
    add = (diff+1)//2
    ops+=add
    n+=add*2
    n=math.isqrt(n)
    ops+=1
    k-=1
  ops +=k
  return ops

c,t = map(int, input().split())
for _ in range(t):
  n, k =map(int , input().split())
  print(min_operation(n,k))

ValueError: not enough values to unpack (expected 2, got 0)

In [4]:
"""
Codeforces Round (Div. 2) 题目：Sqrt Problem

题目描述：
给定整数n，可进行两种操作：
1. n := n + 2（增加2）
2. 如果sqrt(n)是整数，则n := sqrt(n)（开方，获得1分）

求获得k分（即进行k次开方）所需的最少操作数。

关键观察：
1. 只能通过加2来增加值，不能减少
2. 要进行开方，当前值必须是完全平方数
3. 从值m到完全平方数s^2，距离m-s^2必须是偶数（因为只能加2）
4. 如果某个值会循环出现，可以直接计算周期性结果
"""

import sys
sys.setrecursionlimit(10**6)


def solve(n, k):
    """
    使用动态规划+周期检测的优化算法
    
    时间复杂度：通常O(log^2(n) * log(k))，特殊情况O(log(k))
    空间复杂度：O(log^2(n) * log(k))（备忘录）
    
    Args:
        n: 初始值，1 <= n <= 10^18
        k: 需要的开方次数，1 <= k <= 10^18
    
    Returns:
        最少操作数
    """
    
    memo = {}
    
    def min_ops(current, remaining_k):
        """
        计算从current开始，进行remaining_k次开方所需的最少操作数
        
        Args:
            current: 当前值
            remaining_k: 还需进行的开方次数
        
        Returns:
            最少操作数
        """
        
        if remaining_k == 0:
            return 0
        
        # 查询备忘录
        if (current, remaining_k) in memo:
            return memo[(current, remaining_k)]
        
        sqrt_current = int(current ** 0.5)
        
        if sqrt_current * sqrt_current == current:
            # current是完全平方数，直接开方
            result = 1 + min_ops(sqrt_current, remaining_k - 1)
        else:
            # current不是完全平方数，需要加2到最近的完全平方数
            # 检查(sqrt_current + 1)^2和(sqrt_current + 2)^2
            # 选择满足奇偶性且最小的
            
            next_sqrt = sqrt_current + 1
            next_next_sqrt = sqrt_current + 2
            
            candidates = []
            for s in [next_sqrt, next_next_sqrt]:
                sq = s * s
                # 检查是否能通过加2从current到达sq
                if (sq - current) % 2 == 0:
                    candidates.append((sq, s))
            
            # 选择最小的完全平方数
            best_sq, best_sqrt = min(candidates)
            add_ops_count = (best_sq - current) // 2
            
            # 关键优化：检测周期
            # 如果best_sqrt == current，则陷入循环
            if best_sqrt == current:
                # 每次操作都回到current
                # min_ops(current, k) = (add_ops_count + 1) * k
                result = (add_ops_count + 1) * remaining_k
            else:
                # 正常递归
                result = add_ops_count + 1 + min_ops(best_sqrt, remaining_k - 1)
        
        memo[(current, remaining_k)] = result
        return result
    
    return min_ops(n, k)


def main():
    """主程序：读取输入并输出结果"""
    c, t = map(int, input().split())
    
    for _ in range(t):
        n, k = map(int, input().split())
        print(solve(n, k))


# 测试用例验证
if __name__ == "__main__":
    # 如果希望本地测试，取消注释下面的代码
    test_cases = [
        (6, 1),
        (1, 3),
        (14514, 23333),
        (2011112920110906, 1),
        (3, 1919810233114514),
    ]
    
    expected = [
        6,
        3,
        46860,
        15268726,
        7679240932458056,
    ]
    
    print("=== 本地测试 ===")
    all_pass = True
    for i, (n, k) in enumerate(test_cases):
        result = solve(n, k)
        status = "✓ PASS" if result == expected[i] else "✗ FAIL"
        print(f"Test {i+1}: {status}")
        print(f"  Input: n={n}, k={k}")
        print(f"  Result: {result}, Expected: {expected[i]}")
        if result != expected[i]:
            all_pass = False
        print()
    
    print("=" * 40)
    if all_pass:
        print("所有测试用例通过！✓")
    else:
        print("部分测试用例失败！✗")
    
    # 取消下面的注释以从标准输入读取
    # main()

=== 本地测试 ===
Test 1: ✓ PASS
  Input: n=6, k=1
  Result: 6, Expected: 6

Test 2: ✓ PASS
  Input: n=1, k=3
  Result: 3, Expected: 3

Test 3: ✓ PASS
  Input: n=14514, k=23333
  Result: 46860, Expected: 46860

Test 4: ✓ PASS
  Input: n=2011112920110906, k=1
  Result: 15268726, Expected: 15268726

Test 5: ✓ PASS
  Input: n=3, k=1919810233114514
  Result: 7679240932458056, Expected: 7679240932458056

所有测试用例通过！✓


In [ ]:
import sys
sys.setrecursionlimit(10**6)


def solve(n, k):
    memo = {}
    
    def min_ops(current, remaining_k):
        if remaining_k == 0:
            return 0
        if (current, remaining_k) in memo:
            return memo[(current, remaining_k)]
        sqrt_current = int(current ** 0.5)
        if sqrt_current * sqrt_current == current:
            result = 1 + min_ops(sqrt_current, remaining_k - 1)
        else:
            next_sqrt = sqrt_current + 1
            next_next_sqrt = sqrt_current + 2
            candidates = []
            for s in [next_sqrt, next_next_sqrt]:
                sq = s * s
                if (sq - current) % 2 == 0:
                    candidates.append((sq, s))
            best_sq, best_sqrt = min(candidates)
            add_ops_count = (best_sq - current) // 2

            if best_sqrt == current:
                result = (add_ops_count + 1) * remaining_k
            else:
                result = add_ops_count + 1 + min_ops(best_sqrt, remaining_k - 1)
        memo[(current, remaining_k)] = result
        return result
    
    return min_ops(n, k)


c, t = map(int, input().split())
    
for _ in range(t):
    n, k = map(int, input().split())
    print(solve(n, k))

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ll = long long;

struct PairHash {
    size_t operator()(const pair<ll,ll>& p) const {
        return hash<ll>()(p.first) ^ (hash<ll>()(p.second) << 1);
    }
};

unordered_map<pair<ll,ll>, ll, PairHash> memo;

ll min_ops(ll current, ll k){
    if(k == 0) return 0;
    pair<ll,ll> key = {current, k};
    if(memo.count(key)) return memo[key];
    ll sqrt_current = sqrtl(current);
    if(sqrt_current * sqrt_current == current){
        return memo[key] = 1 + min_ops(sqrt_current, k - 1);
    }
    ll next_sqrt = sqrt_current + 1;
    ll next_next_sqrt = sqrt_current + 2;
    vector<pair<ll,ll>> candidates;
    for(ll s : {next_sqrt, next_next_sqrt}){
        ll sq = s * s;
        if((sq - current) % 2 == 0){
            candidates.push_back({sq, s});
        }
    }

    auto best = *min_element(candidates.begin(), candidates.end());
    ll best_sq = best.first;
    ll best_sqrt = best.second;
    ll add_ops = (best_sq - current) / 2;
    ll result;
    
    if(best_sqrt == current){
        result = (add_ops + 1) * k;
    }else{
        result = add_ops + 1 + min_ops(best_sqrt, k - 1);
    }

    return memo[key] = result;
}

int main(){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    ll c, t;
    cin >> c >> t;

    while(t--){
        ll n, k;
        cin >> n >> k;
        cout << min_ops(n, k) << "\n";
    }

    return 0;
}


# T734963 【MX-J25-T2】「Cfz Round 8」Update Problem

## 题目描述

给定一个整数 $n$，你可以对其进行下面的两种操作：

- $n \leftarrow n+1$，即将 $n$ 增加 $1$。
- $n \leftarrow -n$，即将 $n$ 乘上 $-1$。

现在，你需要按照任意顺序进行 $a$ 次第 $1$ 种操作和 $b$ 次第 $2$ 种操作。设操作过程中 $|n|$ 的最大值为 $m$，你需要使 $m$ 的值尽可能小，并求出该最小值。

## 输入格式

**本题包含多组测试数据。**

输入的第一行包含两个非负整数 $c,t$，分别表示测试点编号与测试数据组数。$c=0$ 表示该测试点为样例。

接下来依次输入每组测试数据，对于每组测试数据：

- 共一行，包含三个非负整数 $n,a,b$。

## 输出格式

对于每组测试数据：

- 输出一行，包含一个整数，表示 $m$ 的最小值。

## 输入输出样例 #1

### 输入 #1

```
0 5
0 5 1
0 6 2
0 114 514
250 5000 200
-13831 114514 1919810
```

### 输出 #1

```
2
2
1
250
13831
```

## 说明/提示

### 样例 1 解释

本组样例包含 $5$ 组测试数据。

- 对于第 $1$ 组测试数据，依次进行第 $1,2,1,1,1$ 种操作即可。
- 对于第 $2$ 组测试数据，依次进行第 $1,2,1,1,2,1$ 种操作即可。

### 数据范围

对于所有测试数据，均有：

- $1 \le t \le 10^5$；
- $0 \le |n|,a,b \le 10^9$。

::cute-table{tuack}
| 测试点编号 | $a\le$ | $b\le$ | 特殊性质 |
|:-:|:-:|:-:|:-:|
| $1$ | $10$ | $10$ | AC |
| $2$ | $150$ | $150$ | CE |
| $3$ | $2000$ | $2000$ | ^ |
| $4$ | $10^5$ | $10^5$ | ^ |
| $5$ | $2$ | $10^9$ | 无 |
| $6$ | $10^9$ | $2$ | ^ |
| $7$ | ^ | $10^9$ | B |
| $8$ | ^ | ^ | C |
| $9$ | ^ | ^ | D |
| $10$ | ^ | ^ | 无 |

- 特殊性质 A：保证 $a + b \le 10$。
- 特殊性质 B：保证 $n \ge 0$。
- 特殊性质 C：保证 $n = 0$。
- 特殊性质 D：保证 $n \le 0$。
- 特殊性质 E：保证 $t \le 100$。

In [ ]:
#include <bits/stdc++.h>
using namespace std;
using ll = long long;

int main(){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    ll c, t;
    cin >> c >> t;
    
    while(t--){
        ll n, a, b;
        cin >> n >> a >> b;
        if(b == 0){
            cout << abs(n + a) << "\n";
            continue;
        }
        ll abs_n = abs(n);
        ll term2 = (a + b + 1) / (b + 2);  // ceil(a/(b+2))
        
        ll answer = max(abs_n, term2);
        cout << answer << "\n";
    }
    
    return 0;
}

In [ ]:
import sys
import math

input = sys.stdin.readline

c, t = map(int, input().split())

for _ in range(t):
    n, a, b = map(int, input().split())
    
    if b == 0:
        print(abs(n + a))
    else:
        abs_n = abs(n)
        # 对应 C++ 的 ceil((a + b + 1)/(b + 2))
        term2 = (a + b + 1) // (b + 2)
        print(max(abs_n, term2))

# T734970 【MX-J25-T3】「Cfz Round 8」Juice Problem

## 题目描述

Yuki 的面前有 $n+1$ 个杯子，编号依次为 $0$ 至 $n$。其中，第 $1$ 个到第 $n$ 个杯子的容积与装着的果汁的体积是固定的：第 $i$ 个杯子的容积为 $a_i$，装着的果汁的体积为 $b_i$；而第 $0$ 个杯子的容积为 $10^9$，装着的果汁的体积是不固定的。

Yuki 定义，操作 $i$ 为将第 $i-1$ 个杯子装着的果汁倒入到第 $i$ 个杯子中。若此时第 $i$ 个杯子装着的果汁的体积大于杯子的容积，则果汁会溢出去，直到杯子装着的果汁的体积等于杯子的容积。

现在，Yuki 有 $q$ 次询问，第 $i$ 次询问给出第两个参数 $v_i,p_i$。你需要求出，若第 $0$ 个杯子装着的果汁的体积为 $v_i$，在 Yuki 依次执行操作 $1,2,\dots,p_i$ 后，第 $p_i$ 个杯子装着的果汁的体积为多少。

注意，这些操作不会真的被执行，也就是说询问之间相互独立。

## 输入格式

**本题包含多组测试数据。**

输入的第一行包含两个非负整数 $c,t$，分别表示测试点编号与测试数据组数。$c=0$ 表示该测试点为样例。

接下来依次输入每组测试数据，对于每组测试数据：

- 第一行包含两个非负整数 $n,q$。
- 接下来 $n$ 行，第 $i$ 行包含两个非负整数 $a_i,b_i$。
- 接下来 $q$ 行，第 $i$ 行包含两个非负整数 $v_i,p_i$。

## 输出格式

对于每组测试数据：

- 输出 $q$ 行，第 $i$ 行包含一个整数，表示第 $i$ 次询问的答案。

## 输入输出样例 #1

### 输入 #1

```
0 1
3 3
4 0
9 8
13 8
5 1
0 2
3 3
```

### 输出 #1

```
4
8
13
```

## 输入输出样例 #2

### 输入 #2

```
0 2
5 3
3 1
6 2
9 3
7 2
8 0
4 3
0 4
1 5
2 1
0 0
3 1
5 2
```

### 输出 #2

```
8
7
7
1
```

## 说明/提示

### 样例 1 解释

本组样例包含 $1$ 组测试数据。

对于第 $1$ 次询问：

- 第 $0$ 个杯子装着的果汁的体积为 $5$，将其倒入到第 $1$ 个杯子中后，由于第 $1$ 个杯子的容积为 $4$ 而 $5\gt 4$，果汁会溢出去，因此最终第 $1$ 个杯子装着的果汁的体积为 $4$。

对于第 $2$ 次询问：

- 执行操作 $1$ 后，第 $1$ 个杯子装着的果汁的体积为 $0$；
- 执行操作 $2$ 后，第 $2$ 个杯子装着的果汁的体积为 $8$。

对于第 $3$ 次询问：

- 执行操作 $1$ 后，第 $1$ 个杯子装着的果汁的体积为 $3$；
- 执行操作 $2$ 后，第 $2$ 个杯子装着的果汁的体积为 $9$；
- 执行操作 $3$ 后，第 $3$ 个杯子装着的果汁的体积为 $13$。

### 数据范围

对于所有测试数据，均有：

- $1 \le t \le 3$；
- $1 \le n \le 2\times10^5$，$0 \le q \le 2\times10^5$；
- 对于所有 $1 \le i \le n$，$0 \le b_i \le a_i \le 10^9$；
- 对于所有 $1 \le i \le q$，$0 \le v_i \le 10^9$，$1 \le p_i \le n$。

::cute-table{tuack}

|  测试点编号  |   $n,q \le$   | 特殊性质 |
| :----------: | :-----------: | :------: |
|   $1\sim3$   | $2\times10^3$ |    无    |
|     $4$      | $2\times10^5$ |    AC    |
|   $5\sim8$   | ^ |    A     |
|     $9$      | ^ |    BC    |
|  $10\sim13$  | ^ |    B     |
|   $14,15$    | ^ |    C     |
| $16 \sim 20$ | ^ |    无    |


- 特殊性质 A：对于所有 $1 \le i \lt n$，均有 $a_i \ge a_{i+1}$。
- 特殊性质 B：对于所有 $1 \le i \le n$，均有 $a_i \le a_{i+1}$。
- 特殊性质 C：对于所有 $1 \le i \le n$，均有 $b_i=0$。

In [ ]:
#include <bits/stdc++.h>
using namespace std;
using ll = long long;

int main(){
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int c, t;
    cin >> c >> t;
    
    while(t--){
        int n, q;
        cin >> n >> q;
        vector<ll> a(n), b(n);
        for(int i = 0; i < n; i++){
            cin >> a[i] >> b[i];
        }
        
        for(int query = 0; query < q; query++){
            ll v;
            int p;
            cin >> v >> p;
            
            vector<ll> cups(p + 1);
            cups[0] = v;  // 杯子0
            for(int i = 1; i <= p; i++){
                cups[i] = b[i - 1];  
            }
            
            for(int op = 1; op <= p; op++){
                ll source = cups[op - 1];
                cups[op] = min(a[op - 1], cups[op] + source);
                cups[op - 1] = 0;
            }
            
            cout << cups[p] << "\n";
        }
    }
    
    return 0;
}

# T734962 【MX-J25-T4】「Cfz Round 8」Color Problem

## 题目描述

给定一个 $3\times n$ 的网格图。定义一种染色方案是合法的，当且仅当：

- 每列恰好有一格被染色。
- 相邻两列被染色的格子不同。

其中，每个格子 $(i,j)$ 都有一个参数 $s_{i,j}$：

- 若 $s_{i,j}=\texttt{0}$，则表示该格子必须不染色。
- 若 $s_{i,j}=\texttt 1$，则表示该格子必须染色。
- 若 $s_{i,j}=\texttt ?$，则表示该格子可以染色或不染色。

你需要求出所有合法的染色方案的最大非染色格的连通块面积大小的和，由于答案可能会很大，因此请将答案对 $998,244,353$ 取模。

## 输入格式

**本题包含多组测试数据。**

输入的第一行包含两个非负整数 $c,t$，分别表示测试点编号与测试数据组数。$c=0$ 表示该测试点为样例。

接下来依次输入每组测试数据，对于每组测试数据：

- 第一行包含一个正整数 $n$。
- 接下来三行，第 $i$ 行包含一个长度为 $n$ 的字符串 $s_{i,1},\dots,s_{i,n}$。

## 输出格式

对于每组测试数据：

- 输出一行，包含一个非负整数，表示所有合法的染色方案的最大非染色格的连通块面积大小的和对 $998,244,353$ 取模的结果。

## 输入输出样例 #1

### 输入 #1

```
0 3
1
?
?
?
2
?0
?1
?0
2
??
??
??
```

### 输出 #1

```
5
6
20
```

## 说明/提示

### 样例 1 解释

本组样例包含 $3$ 组测试数据。

- 对于第 $1$ 组测试数据：
  - 若 $(1,1)$ 被染色，则最大非染色格的连通块面积大小为 $2$。
  - 若 $(2,1)$ 被染色，则最大非染色格的连通块面积大小为 $1$。
  - 若 $(3,1)$ 被染色，则最大非染色格的连通块面积大小为 $2$。
  - 所有方案总和为 $(2 + 1 + 2) \bmod 998,244,353 = 5$。
- 对于第 $2$ 组测试数据：
  - 若 $(1,1)$ 被染色，则最大非染色格的连通块面积大小为 $3$。
  - 若 $(3,1)$ 被染色，则最大非染色格的连通块面积大小为 $3$。
  - **注意，$\boldsymbol{(2,1)}$ 被染色的情况不属于合法方案，因为一个染色方案是合法的需要满足相邻两列被染色的格子不同。**
  - 所有方案总和为 $(3 + 3) \bmod 998,244,353 = 6$。

### 数据范围

对于所有测试数据，均有：

- $1 \le t \le 5$；
- $1 \le n \le 300$；
- 对于所有 $1 \le i \le 3$ 和 $1 \le j \le n$，$s_{i,j} \in \{\texttt 0,\texttt 1 ,\texttt ?\}$。

::cute-table{tuack}

| 测试点编号 | $n \le $ | 特殊性质 |
| :--------: | :------: | :------: |
|    $1$     |   $5$    |    无    |
|    $2$     |   $10$   |    ^    |
|    $3$     |   $15$   |    ^    |
|    $4$     |   $20$   |    ^    |
|    $5$     |   $30$   |    ^    |
|    $6$     |   $40$   |    ^    |
|    $7$     |   $60$   |    ^    |
|    $8$     |   $80$   |    ^    |
|    $9$     |  $100$   |    A     |
|    $10$    |  ^   |    B     |
|    $11$    |  ^   |    C     |
|    $12$    |  ^   |    无    |
|    $13$    |  $200$   |    A     |
|    $14$    |  ^   |    B     |
|    $15$    |  ^   |    C     |
|    $16$    |  ^  |    无    |
|    $17$    |  $300$   |    A     |
|    $18$    | ^   |    B     |
|    $19$    |  ^   |    C     |
|    $20$    |  ^  |    无    |

- 特殊性质 A：对于所有 $1 \le i \le 3$ 和 $1 \le j \le n$，保证 $s_{i,j} \ne \texttt ?$。
- 特殊性质 B：对于所有 $1 \le i \le n$，保证 $s_{1,i} = \texttt 0$。
- 特殊性质 C：对于所有 $1 \le i \le 3$ 和 $1 \le j \le n$，若 $(i+j) \bmod 2 = 0$，则有 $s_{i,j} = 0$。

In [ ]:
from collections import deque

MOD = 998244353

def solve_case(n, grid):
    valid_rows = []
    for j in range(n):
        valid = []
        for row in range(3):
            can_color = True
            for i in range(3):
                cell = grid[i][j]
                if i == row:
                    if cell == '0':
                        can_color = False
                        break
                else:
                    if cell == '1':
                        can_color = False
                        break
            if can_color:
                valid.append(row)
        valid_rows.append(valid)
    
    result = 0
    
    def enumerate_colorings(col, prev_row, coloring):
        nonlocal result
        
        if col == n:
            max_comp = find_max_component(n, coloring)
            result = (result + max_comp) % MOD
            return
        
        for row in valid_rows[col]:
            if col == 0 or row != prev_row:
                enumerate_colorings(col + 1, row, coloring + [row])
    
    enumerate_colorings(0, -1, [])
    return result

def find_max_component(n, coloring):
    uncolored = set()
    for j in range(n):
        for i in range(3):
            if i != coloring[j]:
                uncolored.add((i, j))
    
    if not uncolored:
        return 0
    
    visited = set()
    max_size = 0
    
    for start in uncolored:
        if start in visited:
            continue
        queue = deque([start])
        visited.add(start)
        size = 1
        
        while queue:
            i, j = queue.popleft()
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < 3 and 0 <= nj < n and (ni, nj) not in visited:
                    if (ni, nj) in uncolored:
                        visited.add((ni, nj))
                        queue.append((ni, nj))
                        size += 1
        
        max_size = max(max_size, size)
    
    return max_size
line = input().split()
c, t = int(line[0]), int(line[1])

for _ in range(t):
    n = int(input())
    grid = []
    for i in range(3):
        grid.append(input().strip())
    
    answer = solve_case(n, grid)
    print(answer)

In [ ]:
#include <bits/stdc++.h>
using namespace std;

const int MAXN = 305;
const long long MOD = 998244353;

int n;
string grid[3];
vector<int> valid_rows[MAXN];
long long result = 0;

int findMaxComponent(const vector<int>& coloring) {
    int is_uncolored[3][MAXN];
    memset(is_uncolored, 0, sizeof(is_uncolored));
    
    for (int j = 0; j < n; j++) {
        for (int i = 0; i < 3; i++) {
            is_uncolored[i][j] = (i != coloring[j]);
        }
    }
    
    int visited[3][MAXN];
    memset(visited, 0, sizeof(visited));
    
    int max_size = 0;
    
    for (int start_i = 0; start_i < 3; start_i++) {
        for (int start_j = 0; start_j < n; start_j++) {
            if (is_uncolored[start_i][start_j] && !visited[start_i][start_j]) {
                queue<pair<int, int>> q;
                q.push({start_i, start_j});
                visited[start_i][start_j] = 1;
                int size = 1;
                
                while (!q.empty()) {
                    int i = q.front().first;
                    int j = q.front().second;
                    q.pop();
                    

                    if (i > 0 && is_uncolored[i-1][j] && !visited[i-1][j]) {
                        visited[i-1][j] = 1;
                        q.push({i-1, j});
                        size++;
                    }
                    if (i < 2 && is_uncolored[i+1][j] && !visited[i+1][j]) {
                        visited[i+1][j] = 1;
                        q.push({i+1, j});
                        size++;
                    }
                    if (j > 0 && is_uncolored[i][j-1] && !visited[i][j-1]) {
                        visited[i][j-1] = 1;
                        q.push({i, j-1});
                        size++;
                    }
                    if (j < n-1 && is_uncolored[i][j+1] && !visited[i][j+1]) {
                        visited[i][j+1] = 1;
                        q.push({i, j+1});
                        size++;
                    }
                }
                
                max_size = max(max_size, size);
            }
        }
    }
    
    return max_size;
}

void enumerateColorings(int col, int prev_row, vector<int>& coloring) {
    if (col == n) {
        int max_component = findMaxComponent(coloring);
        result = (result + max_component) % MOD;
        return;
    }
    
    for (int row : valid_rows[col]) {
        if (col > 0 && row == prev_row) {
            continue;
        }
        
        coloring.push_back(row);
        enumerateColorings(col + 1, row, coloring);
        coloring.pop_back();
    }
}

void solveCase() {
    cin >> n;
    
    for (int i = 0; i < 3; i++) {
        cin >> grid[i];
    }
    
    result = 0;
    
    for (int j = 0; j < n; j++) {
        valid_rows[j].clear();
        
        for (int row = 0; row < 3; row++) {
            bool can_color = true;
            
            for (int i = 0; i < 3; i++) {
                char cell = grid[i][j];
                
                if (i == row) {
                    if (cell == '0') {  // Must not color
                        can_color = false;
                        break;
                    }
                } else {
                    if (cell == '1') {  // Must color
                        can_color = false;
                        break;
                    }
                }
            }
            
            if (can_color) {
                valid_rows[j].push_back(row);
            }
        }
    }
    
    vector<int> coloring;
    coloring.reserve(n);
    enumerateColorings(0, -1, coloring);
    
    cout << result << "\n";
}

int main() {
    ios_base::sync_with_stdio(false);
    cin.tie(nullptr);
    
    int c, t;
    cin >> c >> t;
    
    for (int i = 0; i < t; i++) {
        solveCase();
    }
    
    return 0;
}